# AO3 Relationship Dynamics Analyzer
### Analytical Workflow

This notebook walks through the full analytical pipeline used in the Streamlit app — from scraping an AO3 work to producing a final verdict on relationship dynamics between characters.

## 1. Setup & Imports

In [ ]:
import requests
import spacy
from bs4 import BeautifulSoup
from collections import deque, Counter
from lexicon import LEXICON

nlp = spacy.load('en_core_web_sm')
print('spaCy model loaded:', nlp.meta['name'])

## 2. Scraping AO3

AO3 requires the `view_adult=true` parameter to access explicit works. We append it to the URL before fetching. The full work text lives inside a `<div class="userstuff">` element, and ship tags are `<a class="tag">` elements.

In [ ]:
def fetch_ao3(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    if "view_adult=true" not in url:
        url += "?view_adult=true&view_full_work=true"
    else:
        url += "&view_full_work=true"

    res = requests.get(url, headers=headers, timeout=60)
    soup = BeautifulSoup(res.text, "html.parser")

    content = soup.find("div", class_="userstuff")
    text = content.get_text(" ") if content else ""

    all_tags = [t.text for t in soup.find_all("a", class_="tag")]
    return text, all_tags

# --- Try it ---
TEST_URL = "https://archiveofourown.org/works/1704125"
text, all_tags = fetch_ao3(TEST_URL)

print(f"Text length: {len(text)} characters")
print(f"Tags found: {all_tags[:10]}")

## 3. Extracting the Main Ship

AO3 uses `/` for romantic pairings and `&` for platonic ones. We filter out orientation tags (M/M, F/F, etc.) and take the first valid romantic pairing as the main CP.

In [ ]:
ORIENTATION_TAGS = {"M/M", "F/F", "M/F", "F/M", "Gen", "Multi", "Other"}

def extract_main_cp(all_tags):
    relationships = [
        t for t in all_tags
        if "/" in t and "&" not in t and t.strip() not in ORIENTATION_TAGS
    ]
    for tag in relationships:
        tag = tag.strip().replace(" / ", "/").replace(" /", "/").replace("/ ", "/")
        parts = tag.split("/")
        if len(parts) == 2 and all(len(p.strip()) > 0 for p in parts):
            return parts[0].strip(), parts[1].strip()
    return None, None

charA, charB = extract_main_cp(all_tags)
print(f"Character A: {charA}")
print(f"Character B: {charB}")

## 4. Explicit Content Detection

Before scoring, we check whether the fic contains sexual content. This affects how heavily we weight certain interaction signals later.

In [ ]:
LIGHT_INTIMACY = ["kiss", "hug", "touch"]
FOREPLAY_HINT  = ["moan", "pant", "whimper", "hard", "wet", "aching"]
EXPLICIT_SEX   = ["cock", "dick", "shaft", "penetrate", "thrust", "fuck", "inside", "deep", "stretch"]

text_lower = text.lower()
light_flag    = any(k in text_lower for k in LIGHT_INTIMACY)
foreplay_flag = any(k in text_lower for k in FOREPLAY_HINT)
explicit_flag = any(k in text_lower for k in EXPLICIT_SEX)
has_sex = explicit_flag or (foreplay_flag and not light_flag)

print(f"Light intimacy detected : {light_flag}")
print(f"Foreplay hints detected : {foreplay_flag}")
print(f"Explicit content detected: {explicit_flag}")
print(f"has_sex flag            : {has_sex}")

## 5. The Lexicon

The core of the analysis is a hand-curated lexicon of words associated with dominant or submissive behavior. Each entry has a weight (1–3) indicating signal strength.

Categories:
- `STRONG_DOM` / `MID_DOM` — active control, physical dominance
- `STRONG_SUB` / `MID_SUB` / `WEAK_SUB` — submission, vulnerability
- `POSITION_PHRASES` / `PHRASAL_VERBS` — positional language
- `SWITCH_SIGNALS` — words suggesting role reversal

In [ ]:
for category, entries in LEXICON.items():
    if isinstance(entries, dict):
        print(f"{category}: {len(entries)} entries — e.g. {list(entries.keys())[:4]}")
    else:
        print(f"{category}: {len(entries)} items (list)")

## 6. Dependency Parsing with spaCy

We use spaCy to parse each sentence and identify:
- **Subject** (`nsubj` / `nsubjpass`) — who is performing or receiving the action
- **Object** (`dobj` / `pobj`) — who is being acted upon
- **Voice** — active vs passive, which flips the interpretation

Example: in *"A pinned B against the wall"*, A is the active subject (DOM signal) and B is the object (SUB signal).

In [ ]:
sample = f"{charA} pinned {charB} against the wall."
doc = nlp(sample)

print(f"Sentence: {sample}\n")
for token in doc:
    print(f"  {token.text:15} | lemma: {token.lemma_:12} | dep: {token.dep_:12} | head: {token.head.text}")

## 7. Scoring Pipeline

For each sentence in the fic:
1. Check for phrasal/positional patterns — assign sub score to the character present
2. For each verb, identify subject/object and voice
3. Look up the verb lemma in the lexicon
4. Apply score adjustments based on category and voice
5. Track `role_record` (dom/sub counts) for switch detection

In [ ]:
def get_short_name(full_name):
    return full_name.split()[0] if full_name else full_name

def name_in_text(full_name, short_name, text):
    return full_name in text or short_name in text

context_window = deque(maxlen=10)

def resolve_subject(token_text, charA, charB):
    if token_text in [charA, charB]:
        context_window.append(token_text)
        return token_text
    if token_text.lower() in ["he", "him", "his"]:
        for item in reversed(context_window):
            if item in [charA, charB]:
                return item
    return None

shortA = get_short_name(charA)
shortB = get_short_name(charB)
score = {charA: 0.0, charB: 0.0}
role_record = {charA: {"dom": 0, "sub": 0}, charB: {"dom": 0, "sub": 0}}
switch_count = 0
doc = nlp(text[:50000])  # cap for demo speed

for sent in doc.sents:
    sent_text = sent.text.lower().replace(" ", "_")

    # Phrasal / positional patterns
    for phrase_dict_name in ["PHRASAL_VERBS", "POSITION_PHRASES"]:
        for phrase, weight in LEXICON.get(phrase_dict_name, {}).items():
            if phrase in sent_text:
                a_present = name_in_text(charA.lower(), shortA.lower(), sent_text)
                b_present = name_in_text(charB.lower(), shortB.lower(), sent_text)
                if a_present and not b_present:
                    score[charA] -= weight
                    role_record[charA]["sub"] += 1
                elif b_present and not a_present:
                    score[charB] -= weight
                    role_record[charB]["sub"] += 1

    # Token-level analysis
    for token in sent:
        lemma = token.lemma_.lower().replace(" ", "_")
        subject, voice, obj = None, None, None

        for child in token.children:
            if child.dep_ in ["nsubj", "nsubjpass"]:
                subject = resolve_subject(child.text, charA, charB)
                voice = "passive" if child.dep_ == "nsubjpass" else "active"
            elif child.dep_ in ["dobj", "pobj"]:
                obj = resolve_subject(child.text, charA, charB)

        if not subject:
            continue

        for category, word_dict in LEXICON.items():
            if not isinstance(word_dict, dict) or lemma not in word_dict:
                continue
            weight = word_dict[lemma]

            if category == "SWITCH_SIGNALS":
                switch_count += weight
            elif "DOM" in category:
                if voice == "active":
                    score[subject] += weight
                    role_record[subject]["dom"] += 1
                    if obj and obj in score:
                        score[obj] -= weight * 0.5
                        role_record[obj]["sub"] += 1
                elif voice == "passive":
                    score[subject] -= weight
                    role_record[subject]["sub"] += 1
            elif "SUB" in category:
                if obj and obj in score:
                    score[obj] -= weight
                    role_record[obj]["sub"] += 1
                else:
                    score[subject] -= weight
                    role_record[subject]["sub"] += 1
            elif "POSITION" in category:
                if voice == "active":
                    score[subject] += weight * 0.8
                    role_record[subject]["dom"] += 1
                elif voice == "passive":
                    score[subject] -= weight * 0.8
                    role_record[subject]["sub"] += 1

        if role_record[charA]["dom"] > 0 and role_record[charA]["sub"] > 0:
            switch_count += 1
        if role_record[charB]["dom"] > 0 and role_record[charB]["sub"] > 0:
            switch_count += 1

print("Raw scores:")
print(f"  {charA}: {score[charA]:.2f}")
print(f"  {charB}: {score[charB]:.2f}")
print(f"\nRole record: {role_record}")
print(f"Switch signals: {switch_count}")

## 8. Normalization & Final Verdict

Raw scores are normalized so they sum to 1 in absolute terms, making them easier to compare across fics of different lengths.

In [ ]:
total = abs(score[charA]) + abs(score[charB]) + 1e-6
score[charA] /= total
score[charB] /= total
delta = abs(score[charA] - score[charB])

print("Normalized scores:")
print(f"  {charA}: {score[charA]:.3f}")
print(f"  {charB}: {score[charB]:.3f}")
print(f"  Delta (confidence): {delta:.3f}")

if switch_count >= 5:
    verdict = "SWITCH likely"
elif delta < 0.15:
    verdict = "Uncertain"
elif score[charA] > score[charB]:
    verdict = f"{charA} TOP"
else:
    verdict = f"{charB} TOP"

print(f"\n>>> Verdict: {verdict}")

## 9. Limitations & Possible Improvements

**Current limitations:**
- Keyword matching has no understanding of context or irony
- Pronoun resolution only handles `he/him/his` and gets confused with similarly-gendered characters
- Short fics produce unreliable results due to sparse signal
- Non-English words and unusual phrasing are missed entirely

**Possible improvements:**
- Fine-tune a BERT-style classifier on labeled fic sentences
- Use sentence embeddings to measure semantic similarity to prototype sentences
- Expand the lexicon with fandom-specific vocabulary
- Add coreference resolution for more accurate pronoun tracking